In [ ]:
import torch

from dataset_loaders import build_data_loaders
from utils.checkpoints import load_ae_from_path, load_from_wandb
from utils.config import DatasetConfig
from utils.visualisation import show, show_comparison

In [ ]:
ae_path = load_from_wandb("autoencoder_flowers102")
ae = load_ae_from_path(ae_path, device=torch.device("mps"))

In [ ]:
dataset_cfg = DatasetConfig(
    name="flowers102",
    channels=3,
    height=64,
    width=64,
    num_classes=102,
)

dataloader, test_loader = build_data_loaders(dataset_cfg, batch_size=64, shuffle_test=True, num_workers=0)


In [ ]:
images, labels = next(iter(test_loader))

with torch.no_grad():
    outputs = ae(images)
    recon = torch.sigmoid(outputs.reconstructed)

show(images, "Originals")
show(recon, "Reconstructions")


In [ ]:
versions = ["v19", "v20", "v21", "v22"]
#show(images, "Originals")
for version in versions:
    ae_tmp_path = load_from_wandb("autoencoder_flowers102", tag=version)
    ae_tmp = load_ae_from_path(ae_tmp_path, device=torch.device("mps"))
    with torch.no_grad():
        outputs = ae_tmp(images)
        recon = torch.sigmoid(outputs.reconstructed)
        show_comparison(images, recon, f"Reconstructions {version}")

In [ ]:
from utils.latent_analysis import *

In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    latents = ae.encode(images)

z0 = latents[0]  # a single (D,) latent vector
latent_traversal_grid(z0, ae.decode, dims=[0, 1, 2, 3], n_steps=7)

emb = plot_umap_embedding(latents, labels)
ratios = plot_pca_scree(latents)
print(cluster_separability_metrics(latents, labels))
#print(linear_probe_accuracy(latents, labels))
corr, link = dimension_dependency_analysis(latents, method="pearson")
groups = suggest_dimension_grouping(link, n_groups=4)  # feed straight into a custom PC region graph
mi_lab = mutual_info_with_labels(latents, labels)


In [ ]:
images, labels = next(iter(test_loader))
with torch.no_grad():
    latents = ae.encode(images)

z0 = latents[0]  # a single (D,) latent vector
dims = range(0, 16)
latent_traversal_grid(z0, ae.decode, dims=dims, n_steps=7)
